# Fraud Detection with XGBoost

Build a fraud classifier for **PaySim financial transactions** using
`data/PS_20174392719_1491204439457_log.csv`. The target is `isFraud`:
**1 = fraud**, **0 = legitimate**.

This notebook contains the complete workflow: inspect the data, prepare features,
train a baseline, search model parameters, examine fraud thresholds, and visualize
the results. All model preparation, training, tuning, and evaluation code is contained here.
The dataset path is `CSV_PATH` in the settings cell. Portfolio context is in
`README.md`, `docs/case-study.md`, and `docs/technical-report.md`.

Method reference: [Stuart Walker's fraud-detection article](https://www.linkedin.com/pulse/fraud-detection-using-xgboost-machine-learning-approach-stuart-walker-apeie/).
The article's final reported fraud precision and recall are 74% and 96% at
threshold 0.9. This notebook calculates its own results rather than prescribing
those scores. Package versions and omitted settings can affect the comparison.


## 1. Prepare the Python environment

Use a **Python 3.12** notebook kernel in Cursor. If a kernel is not available,
install one in your chosen environment with `python -m pip install ipykernel`.
The following cell checks the tested package versions and installs missing or
mismatched packages. It normally runs only once. If it asks you to restart the
kernel, restart and run the notebook from the top.


In [1]:
import sys
import subprocess
from importlib import metadata as package_metadata

if sys.version_info[:2] not in {(3, 11), (3, 12)}:
    raise RuntimeError("Select a Python 3.11 or 3.12 kernel. Python 3.12 is recommended.")

required_versions = {
    "numpy": "2.3.5",
    "pandas": "2.2.3",
    "scipy": "1.17.0",
    "scikit-learn": "1.7.2",
    "xgboost": "3.0.5",
    "plotly": "5.24.1",
    "joblib": "1.5.3",
    "nbformat": "5.10.4",
}
packages_to_install = []
changed_modules_already_loaded = []
for package, version in required_versions.items():
    try:
        installed_version = package_metadata.version(package)
    except package_metadata.PackageNotFoundError:
        installed_version = None
    if installed_version != version:
        packages_to_install.append(f"{package}=={version}")
        module = "sklearn" if package == "scikit-learn" else package
        if module in sys.modules:
            changed_modules_already_loaded.append(module)

if packages_to_install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *packages_to_install])
    if changed_modules_already_loaded:
        raise RuntimeError("Dependencies installed. Restart the kernel, then run from the top.")
    print("Dependencies installed. Ready to continue.")
else:
    print("All required package versions are ready.")


All required package versions are ready.


## 2. Import the libraries

pandas and NumPy handle transaction data. XGBoost trains the classifier.
scikit-learn supplies preprocessing, search, and evaluation tools. Plotly
displays the charts inside the notebook.


In [2]:
import gc
import json
import os
import time
from importlib import metadata as package_metadata
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    average_precision_score, classification_report, confusion_matrix,
    make_scorer, precision_recall_curve, precision_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

required_versions = {
    "numpy": "2.3.5",
    "pandas": "2.2.3",
    "scipy": "1.17.0",
    "scikit-learn": "1.7.2",
    "xgboost": "3.0.5",
    "plotly": "5.24.1",
    "joblib": "1.5.3",
    "nbformat": "5.10.4",
}

pio.templates.default = "plotly_white"
pio.renderers.default = "plotly_mimetype"
pd.options.display.float_format = "{:,.6f}".format

print("Package versions:")
display(pd.Series({name: package_metadata.version(name) for name in required_versions},
                  name="version").to_frame())


Package versions:


,version
numpy,2.3.5
pandas,2.2.3
scipy,1.17.0
scikit-learn,1.7.2
xgboost,3.0.5
plotly,5.24.1
joblib,1.5.3
nbformat,5.10.4


## 3. Set the dataset location and experiment settings

The defaults use the full dataset and perform the complete parameter search.
For a faster installation check, set `SAMPLE_ROWS = 100_000` and
`RUN_PARAMETER_SEARCH = False`. Sample results should not be compared with the
full-dataset benchmark. The full search performs 30 cross-validation fits and
one final refit. Four training threads and sequential search fits limit RAM use.


In [11]:
CSV_PATH = Path("data") / "PS_20174392719_1491204439457_log.csv"
RANDOM_SEED = 42
TEST_FRACTION = 0.20
CPU_THREADS = min(4, os.cpu_count() or 1)
SAMPLE_ROWS = None
RUN_PARAMETER_SEARCH = True
SEARCH_ITERATIONS = 10
FINAL_THRESHOLD = 0.90
SAVE_RESULTS = False
OUTPUT_DIRECTORY = Path("xgboost_results")

TRANSACTION_TYPES = ["CASH_IN", "CASH_OUT", "DEBIT", "PAYMENT", "TRANSFER"]
SCALE_COLUMNS = [
    "amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"
]
INPUT_COLUMNS = ["step", "type", *SCALE_COLUMNS, "isFlaggedFraud"]

if not 0 < FINAL_THRESHOLD < 1:
    raise ValueError("FINAL_THRESHOLD must be strictly between 0 and 1.")
if not 1 <= SEARCH_ITERATIONS <= 27:
    raise ValueError("SEARCH_ITERATIONS must be between 1 and 27.")
if SAMPLE_ROWS is not None and SAMPLE_ROWS < 1:
    raise ValueError("SAMPLE_ROWS must be positive or None.")

experiment_started = time.perf_counter()


## 4. Load the transactions

Load the label and the nine source columns needed for modeling. Account
identifiers (`nameOrig` and `nameDest`) are excluded at reading time to reduce
memory use. This has the same feature-selection effect as dropping them later.


In [12]:
CSV_PATH = Path(CSV_PATH)
if not CSV_PATH.is_file():
    dataset_path = Path("data") / CSV_PATH.name
    if dataset_path.is_file():
        CSV_PATH = dataset_path
if not CSV_PATH.is_file():
    raise FileNotFoundError(
        f"Could not find {CSV_PATH.name}. Put it in data/ or edit CSV_PATH above."
    )

original_columns = pd.read_csv(CSV_PATH, nrows=0).columns.tolist()
required_columns = INPUT_COLUMNS + ["isFraud"]
missing_columns = set(required_columns) - set(original_columns)
if missing_columns:
    raise ValueError(f"Missing CSV columns: {sorted(missing_columns)}")

transactions = pd.read_csv(
    CSV_PATH,
    usecols=required_columns,
    dtype={"type": "category", **{c: "float64" for c in required_columns if c != "type"}},
)
full_dataset_rows = len(transactions)
if SAMPLE_ROWS is not None and SAMPLE_ROWS < full_dataset_rows:
    transactions = transactions.sample(n=SAMPLE_ROWS, random_state=RANDOM_SEED)
transactions = transactions.reset_index(drop=True)

print(f"CSV columns: {len(original_columns)}; loaded modeling columns: {len(required_columns)}")
print(f"Rows used: {len(transactions):,} of {full_dataset_rows:,}")
display(transactions.head())


CSV columns: 11; loaded modeling columns: 9
Rows used: 6,362,620 of 6,362,620


,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1.000000,PAYMENT,"9,839.640000","170,136.000000","160,296.360000",0.000000,0.000000,0.000000,0.000000
1,1.000000,PAYMENT,"1,864.280000","21,249.000000","19,384.720000",0.000000,0.000000,0.000000,0.000000
2,1.000000,TRANSFER,181.000000,181.000000,0.000000,0.000000,0.000000,1.000000,0.000000
3,1.000000,CASH_OUT,181.000000,181.000000,0.000000,"21,182.000000",0.000000,1.000000,0.000000
4,1.000000,PAYMENT,"11,668.140000","41,554.000000","29,885.860000",0.000000,0.000000,0.000000,0.000000


## 5. Check the input data

Check missing values, numeric validity, supported transaction types, and the
binary labels. Zero balances are retained as values from the simulation.


In [13]:
transactions.info(memory_usage="deep")
print("\nMissing values by column:")
display(transactions.isna().sum().to_frame("missing_values"))

if transactions.empty or transactions[required_columns].isna().any().any():
    raise ValueError("The required transaction data must be populated and complete.")
numeric_columns = [c for c in required_columns if c != "type"]
numeric_values = transactions[numeric_columns].to_numpy(dtype=np.float64)
if not np.isfinite(numeric_values).all() or (numeric_values < 0).any():
    raise ValueError("PaySim numeric values must be finite and nonnegative.")
del numeric_values
if not set(transactions["type"].unique()).issubset(TRANSACTION_TYPES):
    raise ValueError("The dataset contains an unsupported transaction type.")
for column in ["isFraud", "isFlaggedFraud"]:
    if not transactions[column].isin([0, 1]).all():
        raise ValueError(f"{column} must contain only 0 or 1.")
if transactions["isFraud"].nunique() != 2:
    raise ValueError("Both fraud and legitimate examples are required.")

print("Input checks passed.")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 9 columns):
 #   Column          Dtype   
---  ------          -----   
 0   step            float64 
 1   type            category
 2   amount          float64 
 3   oldbalanceOrg   float64 
 4   newbalanceOrig  float64 
 5   oldbalanceDest  float64 
 6   newbalanceDest  float64 
 7   isFraud         float64 
 8   isFlaggedFraud  float64 
dtypes: category(1), float64(8)
memory usage: 394.4 MB

Missing values by column:


,missing_values
step,0
type,0
amount,0
oldbalanceOrg,0
newbalanceOrig,0
oldbalanceDest,0
newbalanceDest,0
isFraud,0
isFlaggedFraud,0


Input checks passed.


## 6. Examine the class imbalance

Fraud is rare in this dataset. A model can achieve high accuracy while
detecting no fraud, so fraud precision, recall, F1, and error counts will be
reported alongside accuracy. The bar chart uses a logarithmic count axis.


In [14]:
fraud_counts = transactions["isFraud"].value_counts().sort_index()
class_summary = pd.DataFrame({
    "class": ["legitimate", "fraud"],
    "transactions": [int(fraud_counts[0]), int(fraud_counts[1])],
})
class_summary["share_percent"] = 100 * class_summary["transactions"] / len(transactions)
display(class_summary)
print(f"Always predicting legitimate would achieve {fraud_counts[0] / len(transactions):.4%} accuracy.")

class_figure = px.bar(
    class_summary, x="class", y="transactions", text="transactions",
    log_y=True, title="PaySim class distribution (log count axis)",
)
class_figure.update_traces(texttemplate="%{text:,}", textposition="outside")
class_figure.show()


,class,transactions,share_percent
0,legitimate,6354407,99.870918
1,fraud,8213,0.129082


Always predicting legitimate would achieve 99.8709% accuracy.


## 7. Separate the target from the input features

`isFraud` supplies the training labels and is never a model input. The source
fraud flag remains an input to follow the article's feature set.


In [15]:
raw_features = transactions[INPUT_COLUMNS]
y = transactions["isFraud"].to_numpy(dtype=np.int8)
dataset_description = {
    "filename": CSV_PATH.name,
    "full_dataset_rows": full_dataset_rows,
    "used_rows": len(transactions),
    "fraud_rows": int(y.sum()),
    "fraud_rate": float(y.mean()),
}
print("Source features:", INPUT_COLUMNS)
print("Target: isFraud")


Source features: ['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud']
Target: isFraud


## 8. Encode transaction type and standardize amount/balance columns

Use four type indicators with `CASH_IN` as the reference category. Keep `step`
and `isFlaggedFraud` unscaled. The transformed table contains eleven features.

**Reproduction choice:** the article fits scaling before splitting, so this
notebook does the same. The scaler therefore sees held-out feature statistics.
For a new evaluation, fit preprocessing within training folds and use separate
validation data to choose thresholds. Scaling is retained to reproduce the
workflow; tree models do not require standardization.


In [16]:
preprocessor = ColumnTransformer(
    transformers=[
        ("step", "passthrough", ["step"]),
        ("balances", StandardScaler(), SCALE_COLUMNS),
        ("flag", "passthrough", ["isFlaggedFraud"]),
        ("type", OneHotEncoder(
            categories=[TRANSACTION_TYPES], drop="first", handle_unknown="error",
            sparse_output=False, dtype=np.float64,
        ), ["type"]),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

X = preprocessor.fit_transform(raw_features)
feature_names = preprocessor.get_feature_names_out().tolist()
assert "isFraud" not in feature_names
print("Prepared feature matrix:", X.shape)
display(pd.DataFrame(X[:5], columns=feature_names))


Prepared feature matrix: (6362620, 11)


,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFlaggedFraud,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1.000000,-0.281560,-0.229810,-0.237622,-0.323814,-0.333411,0.000000,0.000000,0.000000,1.000000,0.000000
1,1.000000,-0.294767,-0.281359,-0.285812,-0.323814,-0.333411,0.000000,0.000000,0.000000,1.000000,0.000000
2,1.000000,-0.297555,-0.288654,-0.292442,-0.323814,-0.333411,0.000000,0.000000,0.000000,0.000000,1.000000
3,1.000000,-0.297555,-0.288654,-0.292442,-0.317582,-0.333411,0.000000,1.000000,0.000000,0.000000,0.000000
4,1.000000,-0.278532,-0.274329,-0.282221,-0.323814,-0.333411,0.000000,0.000000,0.000000,1.000000,0.000000


## 9. Split the prepared data into training and test sets

Use the article's unstratified 80/20 split with seed 42. Preserve a small set
of raw held-out examples for the scoring demonstration at the end. The test
rows are excluded from model fitting and hyperparameter search.


In [17]:
train_rows, test_rows = train_test_split(
    np.arange(len(y)), test_size=TEST_FRACTION, random_state=RANDOM_SEED,
)
X_train, X_test = X[train_rows], X[test_rows]
y_train, y_test = y[train_rows], y[test_rows]
if min(np.bincount(y_train, minlength=2)) < 3 or len(np.unique(y_test)) != 2:
    raise ValueError("Too few examples of either class. Increase SAMPLE_ROWS or use all rows.")

example_positions = np.concatenate([
    np.flatnonzero(y_test == 0)[:7], np.flatnonzero(y_test == 1)[:3]
])
example_transactions = raw_features.iloc[test_rows[example_positions]].copy()

split_summary = pd.DataFrame({
    "partition": ["training", "test"],
    "rows": [len(y_train), len(y_test)],
    "fraud_cases": [int(y_train.sum()), int(y_test.sum())],
})
display(split_summary)

del X, raw_features, transactions
_ = gc.collect()


,partition,rows,fraud_cases
0,training,5090096,6593
1,test,1272524,1620


## 10. Calculate the fraud-class weight

`scale_pos_weight` is the training ratio of legitimate transactions to fraud
transactions. It is calculated from the training labels rather than hardcoded
or estimated from the test labels.


In [18]:
negative_training_cases = int((y_train == 0).sum())
positive_training_cases = int((y_train == 1).sum())
scale_pos_weight = negative_training_cases / positive_training_cases
print(f"Legitimate training cases: {negative_training_cases:,}")
print(f"Fraud training cases: {positive_training_cases:,}")
print(f"scale_pos_weight: {scale_pos_weight:.8f}")


Legitimate training cases: 5,083,503
Fraud training cases: 6,593
scale_pos_weight: 771.04550281


## 11. Create and train the baseline classifier

The valid binary-classification objective is `binary:logistic`. Use 100 trees,
depth 6, and learning rate 0.3 for the baseline, with explicit settings for
reproducibility. CPU histogram training handles the full dataset efficiently.


In [19]:
base_model_settings = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "n_estimators": 100,
    "max_depth": 6,
    "learning_rate": 0.3,
    "scale_pos_weight": scale_pos_weight,
    "random_state": RANDOM_SEED,
    "base_score": 0.5,
    "tree_method": "hist",
    "device": "cpu",
    "max_bin": 256,
    "n_jobs": CPU_THREADS,
    "importance_type": "gain",
}
baseline_model = XGBClassifier(**base_model_settings)
baseline_started = time.perf_counter()
baseline_model.fit(X_train, y_train)
baseline_scores = baseline_model.predict_proba(X_test)[:, 1]
print(f"Baseline trained in {time.perf_counter() - baseline_started:.1f} seconds.")


Baseline trained in 6.7 seconds.


## 12. Evaluate the baseline at threshold 0.5

A score at or above the threshold is classified as fraud. In the confusion
matrix, rows are actual labels and columns are predicted labels:
`[[TN, FP], [FN, TP]]`.


In [20]:
def metrics_at_threshold(labels, scores, threshold):
    actual_fraud = np.asarray(labels) == 1
    predicted_fraud = np.asarray(scores) >= threshold
    tp = int(np.sum(actual_fraud & predicted_fraud))
    fp = int(np.sum(~actual_fraud & predicted_fraud))
    fn = int(np.sum(actual_fraud & ~predicted_fraud))
    tn = int(np.sum(~actual_fraud & ~predicted_fraud))
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "threshold": float(threshold), "accuracy": (tp + tn) / len(labels),
        "precision": precision, "recall": recall, "f1": f1,
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
        "alert_rate": (tp + fp) / len(labels),
    }

def display_classification_report(labels, scores, threshold):
    print(classification_report(
        labels, (scores >= threshold).astype(np.int8), labels=[0, 1],
        target_names=["legitimate", "fraud"], digits=4, zero_division=0,
    ))

baseline_metrics = metrics_at_threshold(y_test, baseline_scores, 0.5)
display_classification_report(y_test, baseline_scores, 0.5)
print("Baseline confusion matrix:")
display(pd.DataFrame(
    [[baseline_metrics["tn"], baseline_metrics["fp"]],
     [baseline_metrics["fn"], baseline_metrics["tp"]]],
    index=["actual legitimate", "actual fraud"],
    columns=["predicted legitimate", "predicted fraud"],
))


              precision    recall  f1-score   support

  legitimate     1.0000    0.9985    0.9993   1270904
       fraud     0.4645    0.9895    0.6322      1620

    accuracy                         0.9985   1272524
   macro avg     0.7322    0.9940    0.8157   1272524
weighted avg     0.9993    0.9985    0.9988   1272524

Baseline confusion matrix:


,predicted legitimate,predicted fraud
actual legitimate,1269056,1848
actual fraud,17,1603


## 13. Define the hyperparameter search

Search the article's depth, learning-rate, and tree-count candidates. Use
fraud precision to compare ten sampled combinations under three-fold
cross-validation. The test set is not passed to the search.


In [22]:
parameter_candidates = {
    "max_depth": [4, 6, 8],
    "learning_rate": [0.01, 0.1, 0.2],
    "n_estimators": [50, 100, 200],
    "scale_pos_weight": [scale_pos_weight],
}
parameter_search = RandomizedSearchCV(
    estimator=XGBClassifier(**base_model_settings),
    param_distributions=parameter_candidates,
    n_iter=SEARCH_ITERATIONS,
    scoring=make_scorer(precision_score, zero_division=0),
    cv=3,
    random_state=RANDOM_SEED,
    n_jobs=1,
    verbose=2,
    refit=True,
    error_score="raise",
)
display(parameter_candidates)


{'max_depth': [4, 6, 8],
 'learning_rate': [0.01, 0.1, 0.2],
 'n_estimators': [50, 100, 200],
 'scale_pos_weight': [771.0455028060064]}

## 14. Run the search and obtain the optimized model

The search automatically refits its winning configuration on the complete
training partition. If `RUN_PARAMETER_SEARCH` is false, fit the article's
published configuration directly and clearly identify the skipped search.


In [23]:
search_started = time.perf_counter()
if RUN_PARAMETER_SEARCH:
    parameter_search.fit(X_train, y_train)
    optimized_model = parameter_search.best_estimator_
    best_parameters = parameter_search.best_params_
    best_cv_precision = float(parameter_search.best_score_)
    print("Best parameters:", best_parameters)
    print(f"Best CV fraud precision: {best_cv_precision:.6f}")
else:
    best_parameters = {
        "max_depth": 8, "learning_rate": 0.1,
        "n_estimators": 200, "scale_pos_weight": scale_pos_weight,
    }
    optimized_model = XGBClassifier(**{**base_model_settings, **best_parameters})
    optimized_model.fit(X_train, y_train)
    best_cv_precision = None
    print("Search skipped. Fitted the published parameter configuration:", best_parameters)

print(f"Optimization stage completed in {time.perf_counter() - search_started:.1f} seconds.")


Fitting 3 folds for each of 10 candidates, totalling 30 fits
[CV] END learning_rate=0.01, max_depth=8, n_estimators=200, scale_pos_weight=771.0455028060064; total time=   9.8s
[CV] END learning_rate=0.01, max_depth=8, n_estimators=200, scale_pos_weight=771.0455028060064; total time=   9.8s
[CV] END learning_rate=0.01, max_depth=8, n_estimators=200, scale_pos_weight=771.0455028060064; total time=   9.6s
[CV] END learning_rate=0.1, max_depth=6, n_estimators=100, scale_pos_weight=771.0455028060064; total time=   4.5s
[CV] END learning_rate=0.1, max_depth=6, n_estimators=100, scale_pos_weight=771.0455028060064; total time=   4.5s
[CV] END learning_rate=0.1, max_depth=6, n_estimators=100, scale_pos_weight=771.0455028060064; total time=   4.6s
[CV] END learning_rate=0.1, max_depth=4, n_estimators=50, scale_pos_weight=771.0455028060064; total time=   2.5s
[CV] END learning_rate=0.1, max_depth=4, n_estimators=50, scale_pos_weight=771.0455028060064; total time=   2.5s
[CV] END learning_rate=0.1

## 15. Inspect the search results

Rank candidates by cross-validation precision. These are training-fold
results, so they do not replace the final test evaluation.


In [24]:
if RUN_PARAMETER_SEARCH:
    cv_results = pd.DataFrame(parameter_search.cv_results_).sort_values("rank_test_score")
    display(cv_results[[
        "param_max_depth", "param_learning_rate", "param_n_estimators",
        "mean_test_score", "std_test_score", "rank_test_score",
    ]].reset_index(drop=True))
else:
    cv_results = None
    print("No cross-validation results because parameter search was disabled.")


,param_max_depth,param_learning_rate,param_n_estimators,mean_test_score,std_test_score,rank_test_score
0,8,0.100000,200,0.527224,0.014735,1
1,8,0.200000,50,0.330434,0.019835,2
2,8,0.100000,100,0.297250,0.017591,3
3,6,0.200000,50,0.197633,0.009499,4
4,6,0.100000,100,0.181426,0.008546,5
5,4,0.100000,200,0.177682,0.009841,6
6,8,0.010000,200,0.130671,0.010544,7
7,6,0.100000,50,0.123523,0.004657,8
8,4,0.100000,50,0.076007,0.003400,9
9,4,0.010000,50,0.049728,0.006771,10


## 16. Evaluate the optimized model at threshold 0.5

Compare its fraud precision and recall with the baseline. Error counts
remain useful even when the majority-class report rounds close to 1.00.


In [25]:
optimized_scores = optimized_model.predict_proba(X_test)[:, 1]
optimized_metrics_05 = metrics_at_threshold(y_test, optimized_scores, 0.5)
display_classification_report(y_test, optimized_scores, 0.5)

initial_comparison = pd.DataFrame([
    {"model": "baseline", **baseline_metrics},
    {"model": "optimized", **optimized_metrics_05},
])
display(initial_comparison[["model", "threshold", "accuracy", "precision", "recall", "f1", "fp", "fn"]])


              precision    recall  f1-score   support

  legitimate     1.0000    0.9986    0.9993   1270904
       fraud     0.4789    0.9858    0.6446      1620

    accuracy                         0.9986   1272524
   macro avg     0.7394    0.9922    0.8220   1272524
weighted avg     0.9993    0.9986    0.9989   1272524



,model,threshold,accuracy,precision,recall,f1,fp,fn
0,baseline,0.500000,0.998534,0.464503,0.989506,0.632222,1848,17
1,optimized,0.500000,0.998616,0.478861,0.985802,0.644601,1738,23


## 17. Examine feature importance

Gain importance measures how much splits on each feature improved the
training objective. It is not a causal explanation. In this feature set,
post-transaction balances can strongly influence retrospective classification.


In [26]:
feature_importance = pd.DataFrame({
    "feature": feature_names,
    "gain_importance": optimized_model.feature_importances_,
}).sort_values("gain_importance", ascending=False).reset_index(drop=True)
display(feature_importance)

importance_figure = px.bar(
    feature_importance.sort_values("gain_importance"),
    x="gain_importance", y="feature", orientation="h",
    title="XGBoost feature importance (gain)",
    labels={"gain_importance": "Gain importance", "feature": "Feature"},
)
importance_figure.show()


,feature,gain_importance
0,newbalanceOrig,0.739195
1,type_PAYMENT,0.157163
2,oldbalanceOrg,0.034606
3,type_TRANSFER,0.027681
4,amount,0.016372
5,newbalanceDest,0.008893
6,isFlaggedFraud,0.005169
7,step,0.004065
8,type_CASH_OUT,0.003826
9,type_DEBIT,0.001933


## 18. Visualize the confusion matrix

Display the true counts in each cell. Logarithmic color scaling keeps the
rare-class error counts visible beside the large legitimate-transaction count.


In [27]:
def confusion_figure(labels, scores, threshold):
    metrics = metrics_at_threshold(labels, scores, threshold)
    counts = np.array([[metrics["tn"], metrics["fp"]], [metrics["fn"], metrics["tp"]]])
    annotations = [
        [f"TN: {metrics['tn']:,}", f"FP: {metrics['fp']:,}"],
        [f"FN: {metrics['fn']:,}", f"TP: {metrics['tp']:,}"],
    ]
    figure = go.Figure(go.Heatmap(
        z=np.log10(counts + 1), x=["legitimate", "fraud"], y=["legitimate", "fraud"],
        text=annotations, texttemplate="%{text}", customdata=counts,
        colorscale="Blues", showscale=False,
        hovertemplate="Actual: %{y}<br>Predicted: %{x}<br>Count: %{customdata:,}<extra></extra>",
    ))
    figure.update_layout(
        title=f"Test confusion matrix at threshold {threshold:.2f} (log color scale)",
        xaxis_title="Predicted", yaxis_title="Actual",
    )
    figure.update_yaxes(autorange="reversed")
    return figure

confusion_default = confusion_figure(y_test, optimized_scores, 0.5)
confusion_default.show()


## 19. Compare decision thresholds

Explore scores from threshold 0.1 through 0.95. Raising the threshold usually
reduces alerts and false positives, while also increasing missed fraud.
`FINAL_THRESHOLD = 0.9` was fixed in the settings to compare with the article.
The test table below describes trade-offs; it is not used to automatically
select a threshold. A new business threshold should be chosen on validation data.


In [28]:
threshold_results = pd.DataFrame([
    metrics_at_threshold(y_test, optimized_scores, float(threshold))
    for threshold in np.round(np.arange(0.1, 0.951, 0.05), 2)
])
display(threshold_results[["threshold", "precision", "recall", "f1", "fp", "fn", "alert_rate"]])

threshold_figure = px.line(
    threshold_results, x="threshold", y=["precision", "recall", "f1"],
    title="Test precision, recall, and F1 across fraud thresholds",
)
threshold_figure.add_vline(
    x=FINAL_THRESHOLD, line_dash="dash",
    annotation_text=f"article comparison: {FINAL_THRESHOLD:.2f}",
)
threshold_figure.update_layout(yaxis_title="Fraud-class score", legend_title_text="Metric")
threshold_figure.update_yaxes(range=[0, 1.02])
threshold_figure.show()


,threshold,precision,recall,f1,fp,fn,alert_rate
0,0.100000,0.238715,0.995679,0.385102,5144,7,0.005310
1,0.150000,0.282506,0.993827,0.439951,4089,10,0.004479
2,0.200000,0.318542,0.992593,0.482304,3440,12,0.003967
3,0.250000,0.350371,0.990123,0.517586,2974,16,0.003598
4,0.300000,0.377176,0.989506,0.546167,2647,17,0.003340
5,0.350000,0.404599,0.988272,0.574144,2356,19,0.003110
6,0.400000,0.429415,0.987654,0.598578,2126,20,0.002928
7,0.450000,0.451668,0.986420,0.619620,1940,22,0.002780
8,0.500000,0.478861,0.985802,0.644601,1738,23,0.002621
9,0.550000,0.507477,0.984568,0.669746,1548,25,0.002470


## 20. Plot the precision-recall and ROC curves

Average precision (AP) summarizes precision and recall over the score ranking.
ROC-AUC measures how the scores rank fraudulent versus legitimate transactions.
Both metrics use every test row. PR drawing coordinates are sampled to keep
the file compact. The ROC plot removes redundant horizontal points while
preserving its curve exactly.


In [29]:
average_precision = float(average_precision_score(y_test, optimized_scores))
roc_auc = float(roc_auc_score(y_test, optimized_scores))
pr_precision, pr_recall, _ = precision_recall_curve(y_test, optimized_scores)
drawing_points = np.unique(np.linspace(
    0, len(pr_recall) - 1, min(4000, len(pr_recall)), dtype=int
))
pr_figure = go.Figure(go.Scatter(
    x=pr_recall[drawing_points], y=pr_precision[drawing_points], mode="lines",
    name="XGBoost",
))
pr_figure.update_layout(
    title=f"Test precision-recall curve | AP = {average_precision:.4f}",
    xaxis_title="Recall", yaxis_title="Precision",
)
pr_figure.show()

false_positive_rates, true_positive_rates, _ = roc_curve(y_test, optimized_scores)
roc_changes = np.flatnonzero(np.diff(true_positive_rates) != 0)
roc_corners = np.unique(np.concatenate([
    [0, len(false_positive_rates) - 1], roc_changes, roc_changes + 1
]))
roc_figure = go.Figure(go.Scatter(
    x=false_positive_rates[roc_corners], y=true_positive_rates[roc_corners],
    mode="lines", name="XGBoost",
))
roc_figure.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines", name="Random ranking", line={"dash": "dash"},
))
roc_figure.update_layout(
    title=f"Test ROC curve | AUC = {roc_auc:.6f}",
    xaxis_title="False positive rate", yaxis_title="True positive rate",
)
roc_figure.show()
print(f"Average precision: {average_precision:.6f}; ROC-AUC: {roc_auc:.6f}")


Average precision: 0.963761; ROC-AUC: 0.999106


## 21. Evaluate the final fixed threshold

Report the baseline, the optimized model at 0.5, and the optimized model at
the final threshold. These are measured results. The accompanying article
comparison keeps the article's final reported values separate from this run.


In [30]:
final_metrics = metrics_at_threshold(y_test, optimized_scores, FINAL_THRESHOLD)
display_classification_report(y_test, optimized_scores, FINAL_THRESHOLD)
evaluation_summary = pd.DataFrame([
    {"model": "baseline", **baseline_metrics},
    {"model": "optimized", **optimized_metrics_05},
    {"model": "optimized", **final_metrics},
])
display(evaluation_summary[[
    "model", "threshold", "accuracy", "precision", "recall", "f1", "tn", "fp", "fn", "tp"
]])

confusion_final = confusion_figure(y_test, optimized_scores, FINAL_THRESHOLD)
confusion_final.show()

article_reference = {"accuracy": 0.9996, "precision": 0.74, "recall": 0.96, "f1": 0.83}
article_comparison = pd.DataFrame({
    "metric": list(article_reference),
    "article_threshold_0_9_percent": [100 * article_reference[m] for m in article_reference],
    "this_run_percent": [100 * final_metrics[m] for m in article_reference],
})
article_comparison["difference_percentage_points"] = (
    article_comparison["this_run_percent"] - article_comparison["article_threshold_0_9_percent"]
)
print(f"Article reference threshold: 0.90. This run's threshold: {FINAL_THRESHOLD:.2f}.")
display(article_comparison)


              precision    recall  f1-score   support

  legitimate     0.9999    0.9995    0.9997   1270904
       fraud     0.7221    0.9593    0.8240      1620

    accuracy                         0.9995   1272524
   macro avg     0.8610    0.9794    0.9119   1272524
weighted avg     0.9996    0.9995    0.9995   1272524



,model,threshold,accuracy,precision,recall,f1,tn,fp,fn,tp
0,baseline,0.500000,0.998534,0.464503,0.989506,0.632222,1269056,1848,17,1603
1,optimized,0.500000,0.998616,0.478861,0.985802,0.644601,1269166,1738,23,1597
2,optimized,0.900000,0.999478,0.722119,0.959259,0.823966,1270306,598,66,1554


Article reference threshold: 0.90. This run's threshold: 0.90.


,metric,article_threshold_0_9_percent,this_run_percent,difference_percentage_points
0,accuracy,99.960000,99.947820,-0.012180
1,precision,74.000000,72.211896,-1.788104
2,recall,96.000000,95.925926,-0.074074
3,f1,83.000000,82.396607,-0.603393


## 22. Score individual transactions

This function accepts a transaction table containing the nine original input
columns. It uses the fitted preprocessing and optimized model directly in
this notebook. Labels and account identifiers are never passed to the model.
Class-weighted scores are **not calibrated fraud probabilities**.


In [31]:
def score_transactions(transaction_table, threshold=FINAL_THRESHOLD):
    missing = set(INPUT_COLUMNS) - set(transaction_table.columns)
    if missing:
        raise ValueError(f"Missing transaction columns: {sorted(missing)}")
    inputs = transaction_table[INPUT_COLUMNS].copy()
    if inputs.empty or inputs.isna().any().any():
        raise ValueError("Transaction inputs must be populated and complete.")
    unknown_types = set(inputs["type"].unique()) - set(TRANSACTION_TYPES)
    if unknown_types:
        raise ValueError(f"Unsupported transaction types: {sorted(unknown_types)}")
    numeric = [c for c in INPUT_COLUMNS if c != "type"]
    values = inputs[numeric].to_numpy(dtype=np.float64)
    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError("Numeric transaction inputs must be finite and nonnegative.")
    if not inputs["isFlaggedFraud"].isin([0, 1]).all():
        raise ValueError("isFlaggedFraud must be 0 or 1.")
    if not 0 < threshold < 1:
        raise ValueError("The decision threshold must be strictly between 0 and 1.")
    features = preprocessor.transform(inputs)
    scores = optimized_model.predict_proba(features)[:, 1]
    scored = transaction_table.copy()
    scored["fraud_score"] = scores
    scored["predicted_fraud"] = (scores >= threshold).astype(np.int8)
    return scored

scored_examples = score_transactions(example_transactions)
scored_examples["actual_fraud"] = y_test[example_positions]
display(scored_examples[["type", "amount", "fraud_score", "predicted_fraud", "actual_fraud"]])

np.testing.assert_allclose(
    scored_examples["fraud_score"], optimized_scores[example_positions], atol=1e-6, rtol=0
)
assert sum(final_metrics[key] for key in ["tn", "fp", "fn", "tp"]) == len(y_test)
np.testing.assert_array_equal(
    [[final_metrics["tn"], final_metrics["fp"]], [final_metrics["fn"], final_metrics["tp"]]],
    confusion_matrix(y_test, (optimized_scores >= FINAL_THRESHOLD).astype(np.int8), labels=[0, 1]),
)
print("Checks passed: rescoring is consistent and confusion-matrix counts reconcile.")


,type,amount,fraud_score,predicted_fraud,actual_fraud
3737323,CASH_IN,"330,218.420000",0.000000,0,0
264914,PAYMENT,"11,647.080000",0.000005,0,0
85647,CASH_IN,"152,264.210000",0.000001,0,0
5899326,TRANSFER,"1,551,760.630000",0.000000,0,0
2544263,CASH_IN,"78,172.300000",0.000004,0,0
3494160,PAYMENT,915.130000,0.000007,0,0
2331654,CASH_OUT,"20,603.870000",0.000305,0,0
6303116,CASH_OUT,"93,483.230000",0.992197,1,1
3960303,CASH_OUT,"806,863.300000",0.999954,1,1
2736446,TRANSFER,"4,953,893.080000",0.999996,1,1


Checks passed: rescoring is consistent and confusion-matrix counts reconcile.


## 23. Optionally save the model and result tables

The code and charts remain in this notebook. Set `SAVE_RESULTS = True` in the
settings cell to additionally export the fitted model, preprocessing, and
numeric results. No separate Python script is needed.


In [32]:
if SAVE_RESULTS:
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    optimized_model.save_model(OUTPUT_DIRECTORY / "xgboost_model.json")
    joblib.dump(preprocessor, OUTPUT_DIRECTORY / "preprocessor.joblib")
    evaluation_summary.to_csv(OUTPUT_DIRECTORY / "evaluation.csv", index=False)
    threshold_results.to_csv(OUTPUT_DIRECTORY / "thresholds.csv", index=False)
    feature_importance.to_csv(OUTPUT_DIRECTORY / "feature_importance.csv", index=False)
    article_comparison.to_csv(OUTPUT_DIRECTORY / "article_comparison.csv", index=False)
    if cv_results is not None:
        cv_results.to_csv(OUTPUT_DIRECTORY / "cv_results.csv", index=False)
    run_metadata = {
        "dataset": dataset_description,
        "seed": RANDOM_SEED,
        "test_fraction": TEST_FRACTION,
        "class_weight": scale_pos_weight,
        "best_parameters": best_parameters,
        "best_cv_precision": best_cv_precision,
        "final_threshold": FINAL_THRESHOLD,
        "roc_auc": roc_auc,
        "average_precision": average_precision,
        "preprocessing_fit_scope": "full dataset, following the article",
        "versions": {name: package_metadata.version(name) for name in required_versions},
    }
    (OUTPUT_DIRECTORY / "run_metadata.json").write_text(
        json.dumps(run_metadata, indent=2, default=lambda value: value.item()) + "\n"
    )
    print(f"Model and result tables saved in {OUTPUT_DIRECTORY}.")
else:
    print("Results are displayed in this notebook. Optional file exports are disabled.")

print(f"Experiment completed in {time.perf_counter() - experiment_started:.1f} seconds.")


Results are displayed in this notebook. Optional file exports are disabled.
Experiment completed in 374.7 seconds.


## Interpret the experiment

- **Precision** measures how many alerts correspond to real fraud.
- **Recall** measures how many fraud cases the classifier catches.
- **False positives** are legitimate transactions incorrectly flagged.
- **False negatives** are fraud cases the classifier misses.

The article contains inconsistent threshold descriptions and mislabeled
confusion-matrix counts, so use this notebook's computed counts and reports
when explaining the experiment. Rounded majority-class scores near 1.00 do
not establish that the classifier made zero errors.

### Limitations of this reproduction

- PaySim is synthetic. These results are not evidence from a live payment system.
- In this file, fraud labels occur only in `TRANSFER` and `CASH_OUT`. That is a property of the generator, not coverage of other fraud patterns.
- The PaySim author warns against using `oldbalanceOrg`, `newbalanceOrig`, `oldbalanceDest`, and `newbalanceDest` for fraud detection, because detected fraudulent transactions are cancelled. This reproduction retained those fields.
- Preprocessing was fitted on the full dataset before the train/test split, so the scaler saw held-out feature statistics.
- The random 80/20 split does not establish performance on later activity.
- `FINAL_THRESHOLD = 0.90` is a comparison with the article, not a business-optimized launch threshold.
- Class-weighted scores have not been validated as calibrated fraud probabilities.
- Flag counts are transactions, not unique customers or deduplicated investigations.

### Future work, not yet run

A valid-feature rebuild would drop the warned balance fields, keep only inputs available before a transaction completes, fit preprocessing inside the training data, and evaluate again. A chronological evaluation would hold out later `step` values instead of a random split. Neither experiment has been run. Do not read this notebook as if those checks were completed.
